# 02. Synthetic SIR validation with negative binomial observation and particle smoothing

This notebook reproduces the synthetic validation workflow used to compare particle filtering and particle smoothing under a negative binomial observation model. It supports manuscript Figure 3 and Supplementary Figure S1.


In [ ]:
# Repository path setup
# This cell makes the notebook runnable from either the repository root or the notebooks/ directory.
from pathlib import Path
import os


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "notebooks").exists():
            return candidate
    if current.name == "notebooks":
        return current.parent
    return current


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)

for directory in [
    "data/metro",
    "data/NHIS/2016~2017",
    "data/mobility_factor/2016~2017",
    "data/Rt/2016~2017",
    "data/Rt/2017~2018",
    "data/Rt/2018~2019",
    "data/Rt/2022~2023",
    "figures/mobility_factor",
    "figures/2016~2017",
    "figures/HeatMap",
    "figures/validation",
    "results/validation",
    "results/tables",
]:
    Path(directory).mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
# 음이항 분포
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.stats import nbinom
import math
import time

Ntot = 1e7
dt = 1.0
day = 100
cases = 1_000_000
xi = 1.0 / 4.0
gamma = 1.0 / 5.0
beta_s = 0.3
lambda_i = 100.0
nb_r = 10.0

# Systematic resampling
def systematic_resample(weights):
    N = len(weights)
    positions = (np.arange(N) + np.random.rand()) / N
    cumulative = np.cumsum(weights)
    idx = np.zeros(N, dtype=int)
    i = 0
    j = 0
    while i < N:
        if positions[i] < cumulative[j]:
            idx[i] = j
            i += 1
        else:
            j += 1
    return idx

# SIR ODE (단일 궤적)
def sir_ode_single(t, y, xi, gamma, Ntot, day):

    S, I, R = y

    # β(t) 설정
    beta_t = (1.2 + np.sin(2.0 * np.pi * t / day)) * gamma / xi

    dSdt = - xi * beta_t * S * I / Ntot
    dIdt =   xi * beta_t * S * I / Ntot - gamma * I
    dRdt =   gamma * I

    return [dSdt, dIdt, dRdt]

# SIR ODE (입자 벡터 버전)
def sir_ode_vec(t, y, beta_vec, xi, gamma, Ntot, cases):
    y = y.reshape(3, cases)
    S, I, R = y

    dS = - xi * beta_vec * S * I / Ntot
    dI =   xi * beta_vec * S * I / Ntot - gamma * I
    dR =   gamma * I

    return np.concatenate([dS, dI, dR])

# Particle smoother 
def particle_smoother(Pf_beta, Pf_S, rv_index_pf, xi, gamma, Ntot, day, cases):
    Ps_beta = np.zeros([day + 1, cases])

    # t = day (마지막 시점)
    Ps_sample = np.arange(cases)
    Ps_beta[day, :] = Pf_beta[day, Ps_sample]

    # t = day - 1
    Ps_sample = rv_index_pf[day, Ps_sample]
    Ps_beta[day - 1, :] = Pf_beta[day - 1, Ps_sample]

    # t = day-2, ..., 0 까지 역추적
    for k in range(day - 2, -1, -1):
        Ps_sample = rv_index_pf[k + 1, Ps_sample]
        Ps_beta[k, :] = Pf_beta[k, Ps_sample]

    # SIR: R_t = γ * E[β_t S_t] / (σ N)
    Ps_RPN = xi * np.mean(Ps_beta[1:, :] * Pf_S[1:, :], axis=1) / (gamma * Ntot)
    return Ps_RPN

# Pre-defined Rt 
np.random.seed()
start = time.time()

Pre_S = np.zeros(day + 1)
Pre_I = np.zeros(day + 1)
Pre_R = np.zeros(day + 1)
Pre_confirm = np.zeros(day)
Pre_beta = np.zeros(day)
Pre_RPN = np.zeros(day)

# 초기 조건
Pre_I[0] = lambda_i
Pre_R[0] = 0.0
Pre_S[0] = Ntot - Pre_I[0] - Pre_R[0]
y0 = [Pre_S[0], Pre_I[0], Pre_R[0]]

# t=0~day 구간을 한 번에 적분
t_eval = np.arange(0, day + 1, dt)  # 0,1,...,day
sol = solve_ivp(
    fun=lambda t, y: sir_ode_single(t, y, xi, gamma, Ntot, day),
    t_span=(0.0, float(day)),
    y0=y0,
    t_eval=t_eval,
    method='RK45'
)

Pre_S[:] = sol.y[0, :]
Pre_I[:] = sol.y[1, :]
Pre_R[:] = sol.y[2, :]

# 하루 신규 확진자 수: R(t+1) - R(t)
for k in range(day):
    Pre_confirm[k] = Pre_R[k + 1] - Pre_R[k]
    # 해당 시점 beta(t_k)
    Pre_beta[k] = (1.2 + math.sin(2.0 * math.pi * k / day)) * gamma / xi

# 이론적 Rt: R_t = γ * β(t) * S(t) / (σ N)
Pre_RPN = xi * Pre_beta * Pre_S[1:] / (gamma * Ntot)

# Particle filtering 
random_numbers = np.random.normal(0, beta_s, cases * day)
normal_beta = np.reshape(random_numbers, (day, cases))

Pf_S = np.zeros([day + 1, cases])
Pf_I = np.zeros([day + 1, cases])
Pf_R = np.zeros([day + 1, cases])
Pf_beta = np.zeros([day + 1, cases])

Pf_confirm = np.zeros([day, cases])
Pf_RPN = np.zeros(day)

# 필터용 조상 인덱스 저장
rv_index_pf = np.zeros([day + 1, cases], dtype=int)

# 초기 상태 (I는 포아송, R=0)
Pf_I[0, :] = np.random.poisson(lambda_i, size=cases)
Pf_R[0, :] = 0.0
Pf_S[0, :] = Ntot - Pf_I[0, :] - Pf_R[0, :]

# 초기 beta: R0 ≈ 1.05 * exp(N(0,β_s))
# R0 ≈ γ β0 S0 / (σ N) → β0 ≈ R0 * σ N / (γ S0)
Pf_beta[0, :] = 1.05 * np.exp(normal_beta[0, :]) * gamma * Ntot / (xi * Pf_S[0, :])

for k in range(day):

    # beta 진화 (로그-정규 랜덤 워크)
    Pf_beta[k + 1, :] = Pf_beta[k, :] * np.exp(normal_beta[k, :])

    # 현재 상태를 1D 벡터로 만들기
    y0_vec = np.concatenate([Pf_S[k, :], Pf_I[k, :], Pf_R[k, :]])
    beta_vec = Pf_beta[k + 1, :].copy()

    # t = 0 -> dt 구간 적분 (입자 전체)
    sol_pf = solve_ivp(
        fun=lambda t, y: sir_ode_vec(t, y, beta_vec, xi, gamma, Ntot, cases),
        t_span=(0.0, dt),
        y0=y0_vec,
        t_eval=[dt],
        method='RK45'
    )

    y1_vec = sol_pf.y[:, -1]
    y1 = y1_vec.reshape(3, cases)
    Pf_S[k + 1, :], Pf_I[k + 1, :], Pf_R[k + 1, :] = y1

    # 각 입자의 하루 신규 확진자 수: R_{k+1} - R_k
    Pf_confirm[k, :] = Pf_R[k + 1, :] - Pf_R[k, :]

    # ---- 음이항 분포 likelihood ----
    y_k = int(round(Pre_confirm[k]))
    mu = Pf_confirm[k, :]
    mu = np.clip(mu, 1e-6, None)

    r = nb_r
    p = r / (r + mu)
    weight = nbinom.pmf(y_k, n=r, p=p)
    weight = np.maximum(weight, 1e-300)
    weight = weight / np.sum(weight)

    # 리샘플링 + 조상 인덱스 저장
    randomList = systematic_resample(weight)
    rv_index_pf[k + 1, :] = randomList

    Pf_S[k + 1, :] = Pf_S[k + 1, randomList]
    Pf_I[k + 1, :] = Pf_I[k + 1, randomList]
    Pf_R[k + 1, :] = Pf_R[k + 1, randomList]
    Pf_beta[k + 1, :] = Pf_beta[k + 1, randomList]

# 필터링 Rt: R_t = γ * E[β_t S_t] / (σ N)
Pf_RPN = xi * np.mean(Pf_beta[1:, :] * Pf_S[1:, :], axis=1) / (gamma * Ntot)

# Particle smoother
Ps_RPN = particle_smoother(Pf_beta, Pf_S, rv_index_pf, xi, gamma, Ntot, day, cases)

end = time.time()
print("Computation time = " + str(end - start) + " seconds")

# Plot
# plt.figure(figsize=(12,8))
plt.plot(Pre_RPN, 'o--', markersize=3, mfc='None', mec='b', label='Pre-defined R(t)')
plt.plot(Pf_RPN, 'o--', markersize=3, mfc='None', mec='r', label='Particle filtering R(t)')
plt.plot(Ps_RPN, 'o--', markersize=3, mfc='None', mec='g', label='Particle smoother R(t)')
plt.grid(True)
plt.yticks([0.5, 1.0, 1.5, 2.0])
plt.legend(loc=0)
plt.title('SIR Mathematical Model Negative binomial distribution')
plt.xlabel('Time')
plt.ylabel('R(t)')

plt.savefig('figures/validation/SIR Mathematical Model Negative binomial distribution.eps', format='eps', bbox_inches='tight')

plt.show()

In [ ]:
start_day = 0   # 전체 기간 (Day 1 ~ 100)
stable_day = 10 # 초기 불안정 구간 제외 (Day 11 ~ 100)

# 전체 기간 (Day 1 ~ 100) RMSE
rmse_pf_total = np.sqrt(np.mean((Pre_RPN[start_day:] - Pf_RPN[start_day:])**2))
rmse_ps_total = np.sqrt(np.mean((Pre_RPN[start_day:] - Ps_RPN[start_day:])**2))

# 안정화 (Day 11 ~ 100) RMSE
rmse_pf_stable = np.sqrt(np.mean((Pre_RPN[stable_day:] - Pf_RPN[stable_day:])**2))
rmse_ps_stable = np.sqrt(np.mean((Pre_RPN[stable_day:] - Ps_RPN[stable_day:])**2))

print("=" * 40)
print(f"기간: Day {start_day+1} ~ {day}")
print(f"RMSE (Filter)   : {rmse_pf_total:.5f}")
print(f"RMSE (Smoother) : {rmse_ps_total:.5f}") 
print("-" * 40)
print(f"기간: Day {stable_day+1} ~ {day} (초기값 제외)")
print(f"RMSE (Filter)   : {rmse_pf_stable:.5f}")
print(f"RMSE (Smoother) : {rmse_ps_stable:.5f}")
print("=" * 40)

In [ ]:
import pandas as pd

# 각 일자별 오차 계산
daily_rmse_pf = np.sqrt((Pre_RPN - Pf_RPN) ** 2)
daily_rmse_ps = np.sqrt((Pre_RPN - Ps_RPN) ** 2)

# 수치 확인
print(f"{'Day':^5} | {'PF Error':^12} | {'PS Error':^12}")
print("-" * 35)
for t in range(0, day, 1):
    print(f"{t+1:^5d} | {daily_rmse_pf[t]:.6f}     | {daily_rmse_ps[t]:.6f}")
print("-" * 35)

df_results = pd.DataFrame({
    'Day': np.arange(1, day + 1),
    'PF_Error': daily_rmse_pf,
    'PS_Error': daily_rmse_ps
})

df_results.to_csv('results/validation/daily_rmse_results.csv', index=False)

plt.plot(daily_rmse_pf, 'o--', markersize=3, mfc='None', mec='b', label='daily_rmse_pf')
plt.plot(daily_rmse_ps, 'o--', markersize=3, mfc='None', mec='r', label='daily_rmse_ps')
plt.grid(True)
plt.legend(loc=0)
plt.title('SIR Mathematical Model Negative binomial distribution RMSE')
plt.xlabel('Time')
plt.ylabel('RMSE')
plt.yticks([0, 0.05, 0.1, 0.15])
plt.savefig('figures/validation/SIR Mathematical Model Negative binomial distribution RMSE.eps', format='eps', bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv("results/validation/daily_rmse_results.csv")

df